# TikHarm three-modality fusion: video + audio + text

Same frozen embeddings and creator-grouped split as before, now with **speech transcripts and on-screen text** as a third modality.
Small heads only. Each result is the mean of 5 seeds, with 95% bootstrap intervals and paired differences against video-only.

| Method | Idea |
|---|---|
| video / audio / text only | single-modality heads |
| early V+A, V+T, V+A+T | concatenate pooled features (separates what audio and text each add) |
| late fusion | combine per-modality probabilities, weights tuned on val |
| **gated V+A+T (router)** | one learned gate per extra modality, driven by what that modality contains (music/speech scores, text counts) |
| cross-attention V+A+T | video tokens attend to audio tokens and text tokens (a null token keeps missing modalities safe) |

**Controls at test time:** real, audio removed, text removed, both removed, audio from a different-class clip, text from a different-class clip.
**Needs:** `emb/` (video + audio chunks, `order.csv`), `prep/manifest_full.csv`, and `text/text_features.npz` from the text notebook.

In [2]:
# Cell 1: load features, split, near-duplicate exclusion
import os, re, glob, json, time
import numpy as np, pandas as pd
import torch, torch.nn as nn
from sklearn.metrics import f1_score, roc_auc_score, recall_score

from sklearn.model_selection import StratifiedGroupKFold

def creator_of(path):
    b = os.path.splitext(os.path.basename(path))[0]; m = re.match(r"^(.*)_(\d{12,20})$", b)
    return (m.group(1) if m else b).lower()

def duration_match(d, bins, seed=0):
    d = d.copy(); d["dbin"] = pd.cut(d.dur, bins, labels=False, include_lowest=True); parts = []
    for (sp, b), g in d.groupby(["split", "dbin"]):
        c = g.label.value_counts()
        if len(c) < 4: continue
        n_ = int(c.min()); parts += [g[g.label == k].sample(n_, random_state=seed) for k in range(4)]
    return pd.concat(parts) if parts else d.iloc[:0]

def ensure_grouped(order, man, bins=(0, 10, 20, 40, 90, 1e9), seed=0, force=False):
    """If any creator appears in both train/val and test, apply the deterministic creator-grouped re-split
    (same code and seed as the baseline, so every notebook gets the identical split). Row order is unchanged."""
    if "path" not in man:
        print("manifest has no path column: cannot check creators"); return order
    cr = man.path.map(creator_of).values; sp = order.split.values
    if not force and not (set(cr[sp == "test"]) & set(cr[np.isin(sp, ["train", "val"])])):
        print("creator check passed: the split is already creator-grouped"); return order
    print(f"applying the creator-grouped re-split (seed {seed})" if force else "OFFICIAL split detected (creators overlap): applying the creator-grouped re-split (seed 0)")
    y = order.label.values
    rest, test = next(iter(StratifiedGroupKFold(5, shuffle=True, random_state=seed).split(np.zeros(len(order)), y, cr)))
    tr_i, va_i = next(iter(StratifiedGroupKFold(8, shuffle=True, random_state=seed + 1).split(np.zeros(len(rest)), y[rest], cr[rest])))
    new = np.array(["train"] * len(order), dtype=object); new[rest[va_i]] = "val"; new[test] = "test"
    matched = duration_match(pd.DataFrame({"uid": order.uid.values, "dur": man.dur.values, "label": y, "split": new}), list(bins), seed)
    o2 = order.copy(); o2["split"] = new; o2["in_matched"] = o2.uid.isin(matched.uid).astype(int)
    assert not (set(cr[new == "test"]) & set(cr[new != "test"])), "re-split failed"
    print(pd.crosstab(o2.cls, o2.split)); return o2

EMB = os.environ.get("EMB_DIR", "/kaggle/input/datasets/sanawerb/emb-folder/kaggle/working/emb")
PREP = os.environ.get("PREP_DIR", "/kaggle/input/datasets/sanawerb/preparation")
WORK = os.environ.get("WORK_DIR", "/kaggle/working")
RES = f"{WORK}/results"; os.makedirs(RES, exist_ok=True)
CFG = dict(d=256, drop=0.4, mod_drop=0.3, epochs=int(os.environ.get("EPOCHS", 40)), lr=1e-3, wd=0.05, bs=64, smooth=0.1,
           seeds=list(range(int(os.environ.get("NSEEDS", 5)))), nboot=int(os.environ.get("NBOOT", 1000)))
CLASS_NAMES = ["Safe", "Harmful", "Adult", "Suicide"]
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev)

order = pd.read_csv(f"{EMB}/order.csv")
files = sorted(f for f in glob.glob(f"{EMB}/chunk_*.npz") if re.search(r"chunk_\d{3}\.npz$", f))
parts = [np.load(f, allow_pickle=True) for f in files]
Fz = {k: np.concatenate([p[k] for p in parts]) for k in parts[0].files}
assert list(Fz["uid"]) == list(order.uid[: len(Fz["uid"])]), "chunks do not match order.csv"
N = len(Fz["uid"]); order = order.iloc[:N].copy()
man = pd.read_csv(f"{PREP}/manifest_full (1).csv").set_index("uid").loc[order.uid].reset_index()
order = ensure_grouped(order, man)

V = Fz["vit"].astype(np.float32)               # (N, 8, 768)
A = Fz["ast"].astype(np.float32)               # (N, 5, 768)
aud_ok = Fz["aud_ok"].astype(np.float32)
music_p = man["music_p"].values.astype(np.float32) if "music_p" in man else np.zeros(N, np.float32)
speech_p = man["speech_p"].values.astype(np.float32) if "speech_p" in man else np.zeros(N, np.float32)
G = np.stack([music_p, speech_p, aud_ok], 1).astype(np.float32)           # gate inputs
y = order.label.values; split = order.split.values.copy(); in_matched = order.in_matched.values.astype(bool)

# exclude test clips with a near-identical train/val neighbour (cosine > 0.95 on ViT mean embeddings)
vm = V.mean(1); vm /= np.linalg.norm(vm, axis=1, keepdims=True) + 1e-8
te_i, tr_i = np.where(split == "test")[0], np.where(split != "test")[0]
leak = te_i[(vm[te_i] @ vm[tr_i].T).max(1) > 0.95]
split[leak] = "excluded"; print("excluded near-duplicate test clips:", len(leak))


def creator_of(path):
    b = os.path.splitext(os.path.basename(path))[0]; m = re.match(r"^(.*)_(\d{12,20})$", b)
    return (m.group(1) if m else b).lower()

def assert_creator_disjoint(paths, split_arr):
    cr = pd.Series(paths).map(creator_of).values
    shared = set(cr[split_arr == "test"]) & set(cr[np.isin(split_arr, ["train", "val"])])
    assert not shared, (f"{len(shared)} creators appear in both train/val and test. This is the OFFICIAL split, not the "
                        "creator-grouped one. Restore the grouped order.csv (re-split cell) before running.")
    print("creator check passed: no creator appears in both train/val and test")
if "path" in man: assert_creator_disjoint(man.path.values, split)

grp = np.full(N, "other audio", dtype=object)
if "music_dominant" in man: grp[man.music_dominant.values == 1] = "music-dominant"
grp[(speech_p > 0.5) & (grp != "music-dominant")] = "speech present"
print(pd.crosstab(order.cls, [split, in_matched]))
print(pd.Series(grp).value_counts().to_dict())

device: cuda
OFFICIAL split detected (creators overlap): applying the creator-grouped re-split (seed 0)
split            test  train  val
cls                              
Adult Content     218    662   68
Harmful Content   182    669  128
Safe              233    684   70
Suicide           238    669   70
excluded near-duplicate test clips: 1
creator check passed: no creator appears in both train/val and test
col_0           excluded  test       train         val      
col_1              True  False True  False True  False True 
cls                                                         
Adult Content          1   119    98   411   251    42    26
Harmful Content        0    83    99   418   251   102    26
Safe                   0   134    99   433   251    44    26
Suicide                0   139    99   418   251    44    26
{'music-dominant': 2117, 'speech present': 1292, 'other audio': 482}


In [4]:
# Cell 2: metrics, bootstrap, paired differences
def metrics(yt, P):
    pred = P.argmax(1)
    m = dict(macro_F1=f1_score(yt, pred, average="macro"),
             macro_AUC=roc_auc_score(yt, P, multi_class="ovr", average="macro", labels=[0, 1, 2, 3]),
             top2_acc=np.mean([yt[i] in np.argsort(-P[i])[:2] for i in range(len(yt))]))
    rec = recall_score(yt, pred, average=None, labels=[0, 1, 2, 3], zero_division=0)
    m.update({f"rec_{c}": r for c, r in zip(CLASS_NAMES, rec)})
    return m

def with_ci(yt, P, nboot=None):
    nboot = nboot or CFG["nboot"]; pt = metrics(yt, P); rng = np.random.default_rng(0); b = []
    for _ in range(nboot):
        ix = rng.integers(0, len(yt), len(yt))
        if len(np.unique(yt[ix])) == 4: b.append(metrics(yt[ix], P[ix]))
    b = pd.DataFrame(b)
    return pt, {k: (np.percentile(b[k], 2.5), np.percentile(b[k], 97.5)) for k in pt}

def fmt(pt, ci, k): return f"{pt[k]:.3f} [{ci[k][0]:.3f}, {ci[k][1]:.3f}]"

def paired(yt, Pbase, Pother, nboot=None):
    nboot = nboot or CFG["nboot"]; rng = np.random.default_rng(1); d = []
    for _ in range(nboot):
        ix = rng.integers(0, len(yt), len(yt))
        d.append(f1_score(yt[ix], Pother[ix].argmax(1), average="macro") - f1_score(yt[ix], Pbase[ix].argmax(1), average="macro"))
    d = np.array(d); return d.mean(), np.percentile(d, 2.5), np.percentile(d, 97.5)

# ---- text modality (from the text extraction notebook) ----
TEXT_DIR = "/kaggle/input/datasets/sanawerb/text-new/kaggle/working/text"
TF = np.load(f"{TEXT_DIR}/text_features.npz", allow_pickle=True)
assert list(TF["uid"]) == list(order.uid), "text_features.npz does not match order.csv (it must come from the same clip order)"
Tk = np.stack([TF["asr_emb"], TF["ocr_emb"]], 1).astype(np.float32)               # (N, 2, 384): speech, on-screen text
Mt = np.stack([TF["has_asr"], TF["has_ocr"]], 1).astype(np.float32)               # which text sources exist
Gt = np.stack([TF["has_asr"], TF["has_ocr"], np.minimum(TF["asr_words"], 50) / 50,
               np.minimum(TF["ocr_chars"], 100) / 100], 1).astype(np.float32)     # text gate inputs
tgrp = np.where(Mt.sum(1) == 2, "speech + on-screen text", np.where(Mt.sum(1) == 1, "one text source", "no text")).astype(object)
print("text availability:", pd.Series(tgrp).value_counts().to_dict())

text availability: {'one text source': 1880, 'speech + on-screen text': 1103, 'no text': 908}


In [5]:
# Cell 3: the models
class Net3(nn.Module):
    """mode: video | audio | text | early | gate | xattn.  use: which extra modalities a fusion model sees ('A', 'T' or 'AT')."""
    def __init__(s, mode, use="", d=256, p=0.4, n_cls=4):
        super().__init__(); s.mode, s.use = mode, use
        mk = lambda i: nn.Sequential(nn.LayerNorm(i), nn.Dropout(p * 0.5), nn.Linear(i, d), nn.GELU(), nn.Dropout(p))
        s.vp, s.ap, s.tp = mk(768), mk(768), mk(384)
        s.type_emb = nn.Parameter(torch.zeros(2, d))                      # tells speech tokens from on-screen-text tokens
        s.ga = nn.Sequential(nn.Linear(3, 16), nn.GELU(), nn.Linear(16, 1))
        s.gt = nn.Sequential(nn.Linear(4, 16), nn.GELU(), nn.Linear(16, 1))
        s.null = nn.Parameter(torch.zeros(1, 1, d))
        s.attn = nn.MultiheadAttention(d, 4, dropout=0.1, batch_first=True); s.an = nn.LayerNorm(d)
        k = len(use)
        din = {"video": d, "audio": d, "text": d, "early": d * (1 + k) + k, "gate": d * (1 + k), "xattn": d}[mode]
        s.head = nn.Sequential(nn.Dropout(p), nn.Linear(din, n_cls))

    def text_tokens(s, t): return s.tp(t) + s.type_emb                    # (B,2,d)
    def text_pool(s, t, mt):
        return (s.text_tokens(t) * mt[..., None]).sum(1) / mt.sum(1, keepdim=True).clamp(min=1)
    def gate_a(s, ma, ga): return torch.sigmoid(s.ga(ga)).squeeze(-1) * ma
    def gate_t(s, mt, gt): return torch.sigmoid(s.gt(gt)).squeeze(-1) * (mt.sum(1) > 0).float()

    def forward(s, v, a, ma, ga, t, mt, gt):
        if s.mode == "audio": return s.head(s.ap(a).mean(1) * ma[:, None])
        if s.mode == "text": return s.head(s.text_pool(t, mt))
        hv = s.vp(v)
        if s.mode == "video": return s.head(hv.mean(1))
        parts, anyt = [hv.mean(1)], (mt.sum(1) > 0).float()
        if s.mode == "early":
            if "A" in s.use: parts += [s.ap(a).mean(1) * ma[:, None], ma[:, None]]
            if "T" in s.use: parts += [s.text_pool(t, mt), anyt[:, None]]
            return s.head(torch.cat(parts, 1))
        if s.mode == "gate":
            if "A" in s.use: parts += [s.ap(a).mean(1) * s.gate_a(ma, ga)[:, None]]
            if "T" in s.use: parts += [s.text_pool(t, mt) * s.gate_t(mt, gt)[:, None]]
            return s.head(torch.cat(parts, 1))
        kv, keep = [s.null.expand(len(v), 1, -1)], [torch.ones(len(v), 1, dtype=torch.bool, device=v.device)]   # xattn
        if "A" in s.use: kv.append(s.ap(a)); keep.append((ma > 0)[:, None].expand(-1, a.shape[1]))
        if "T" in s.use: kv.append(s.text_tokens(t)); keep.append(mt > 0)
        kv, keep = torch.cat(kv, 1), torch.cat(keep, 1)
        att, _ = s.attn(hv, kv, kv, key_padding_mask=~keep)               # the null token is always visible, so no all-masked rows
        return s.head(s.an(hv + att).mean(1))

    @torch.no_grad()
    def gates(s, ma, ga, mt, gt): return s.gate_a(ma, ga), s.gate_t(mt, gt)

def make_data(mask):
    idx = np.where(mask)[0]; sp = split[idx]; tr = idx[sp == "train"]
    mu_v, sd_v = V[tr].reshape(-1, 768).mean(0), V[tr].reshape(-1, 768).std(0) + 1e-6
    ta = tr[aud_ok[tr] == 1]; mu_a, sd_a = A[ta].reshape(-1, 768).mean(0), A[ta].reshape(-1, 768).std(0) + 1e-6
    tt = Tk[tr][Mt[tr] > 0]; mu_t, sd_t = tt.mean(0), tt.std(0) + 1e-6                  # statistics over present text tokens only
    T = lambda x: torch.tensor(x, dtype=torch.float32, device=dev)
    Tn = ((Tk - mu_t) / sd_t) * Mt[..., None]
    D = dict(V=T((V - mu_v) / sd_v), A=T((A - mu_a) / sd_a), Ma=T(aud_ok), Ga=T(G), Tk=T(Tn), Mt=T(Mt), Gt=T(Gt),
             y=torch.tensor(y, device=dev), y_np=y)
    for s_ in ("train", "val", "test"):
        D[s_] = torch.tensor(idx[sp == s_], device=dev); D[s_ + "_np"] = idx[sp == s_]
    rng = np.random.default_rng(123); te = D["test_np"]; yt = y[te]                     # "wrong" partner: a clip of a different class
    D["wrong"] = torch.tensor([rng.choice(np.where(yt != yt[i])[0]) for i in range(len(te))], device=dev)
    return D

def batch(D, ix, ma=None, mt=None):
    return (D["V"][ix], D["A"][ix], D["Ma"][ix] if ma is None else ma, D["Ga"][ix], D["Tk"][ix],
            D["Mt"][ix] if mt is None else mt, D["Gt"][ix])

def train_one(mode, use, D, seed, mod_drop):
    torch.manual_seed(seed); np.random.seed(seed)
    net = Net3(mode, use, CFG["d"], CFG["drop"]).to(dev)
    tr = D["train"]; ytr = D["y"][tr]
    cnt = torch.bincount(ytr, minlength=4).float(); w = (len(ytr) / (4 * cnt.clamp(min=1))).to(dev)
    opt = torch.optim.AdamW(net.parameters(), lr=CFG["lr"], weight_decay=CFG["wd"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG["epochs"] * int(np.ceil(len(tr) / CFG["bs"])))
    lossf = nn.CrossEntropyLoss(weight=w, label_smoothing=CFG["smooth"])
    for ep in range(CFG["epochs"]):
        net.train(); perm = torch.randperm(len(tr), device=dev)
        for i in range(0, len(tr), CFG["bs"]):
            ix = tr[perm[i:i + CFG["bs"]]]; ma, mt = D["Ma"][ix], D["Mt"][ix]
            if mod_drop > 0 and mode in ("early", "gate", "xattn"):                 # hide each used modality at random
                if "A" in use: ma = ma * (torch.rand(len(ix), device=dev) >= mod_drop).float()
                if "T" in use: mt = mt * (torch.rand(len(ix), device=dev) >= mod_drop).float()[:, None]
            loss = lossf(net(*batch(D, ix, ma, mt)), D["y"][ix])
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    return net.eval()

@torch.no_grad()
def predict(net, D, part, cond="real"):
    ix = D[part]; v, a, ma, ga, t, mt, gt = batch(D, ix)
    if cond in ("shufA", "shufT"):                                                  # take that modality from a different-class clip
        j = D["wrong"]
        if cond == "shufA": a, ma, ga = a[j], ma[j], ga[j]
        else: t, mt, gt = t[j], mt[j], gt[j]
    if cond in ("noA", "noAT"): ma = torch.zeros_like(ma)
    if cond in ("noT", "noAT"): mt = torch.zeros_like(mt)
    return torch.softmax(net(v, a, ma, ga, t, mt, gt), 1).cpu().numpy()

@torch.no_grad()
def gate_values(net, D):
    ix = D["test"]; _, _, ma, ga, _, mt, gt = batch(D, ix)
    ga_, gt_ = net.gates(ma, ga, mt, gt); return ga_.cpu().numpy(), gt_.cpu().numpy()

In [6]:
# Cell 4: smoke test (fails fast if something is wrong)
for mode, use in (("video", ""), ("audio", ""), ("text", ""), ("early", "A"), ("early", "T"), ("early", "AT"),
                  ("gate", "AT"), ("xattn", "A"), ("xattn", "T"), ("xattn", "AT")):
    net = Net3(mode, use).to(dev); B = 6
    v, a, t = torch.randn(B, 8, 768, device=dev), torch.randn(B, 5, 768, device=dev), torch.randn(B, 2, 384, device=dev)
    ga, gt = torch.rand(B, 3, device=dev), torch.rand(B, 4, device=dev)
    for ma, mt in ((1, 1), (0, 0), (1, 0), (0, 1)):                                  # every availability pattern
        out = net(v, a, torch.full((B,), float(ma), device=dev), ga, t, torch.full((B, 2), float(mt), device=dev), gt)
        assert out.shape == (B, 4) and torch.isfinite(out).all(), (mode, use, ma, mt)
        out.sum().backward()
print("all model variants run forward and backward for every audio/text availability pattern")

all model variants run forward and backward for every audio/text availability pattern


In [7]:
# Cell 5: run all methods on one set
SPECS = [("video only", "video", "", 0), ("audio only", "audio", "", 0), ("text only", "text", "", 0),
         ("early V+A", "early", "A", CFG["mod_drop"]), ("early V+T", "early", "T", CFG["mod_drop"]),
         ("early V+A+T", "early", "AT", CFG["mod_drop"]), ("gated V+A+T (router)", "gate", "AT", CFG["mod_drop"]),
         ("cross-attention V+A+T", "xattn", "AT", CFG["mod_drop"]), ("gated V+A+T, no mod-drop", "gate", "AT", 0)]

def conds_for(mode, use):
    if mode == "video": return ["real"]
    if mode == "audio": return ["real", "shufA"]
    if mode == "text": return ["real", "shufT"]
    c = ["real"]
    if "A" in use: c += ["noA", "shufA"]
    if "T" in use: c += ["noT", "shufT"]
    if use == "AT": c += ["noAT"]
    return c

def softmax_np(lp):
    e = np.exp(lp - lp.max(1, keepdims=True)); return e / e.sum(1, keepdims=True)

def run_set(name, mask):
    D = make_data(mask); te = D["test_np"]; yt = y[te]
    print(f"\n##### {name}: train={len(D['train'])} val={len(D['val'])} test={len(te)} #####")
    P, gv, t0 = {}, {}, time.time()
    for label, mode, use, md in SPECS:
        conds = conds_for(mode, use)
        acc = {("real", "val"): []}; acc.update({(c, "test"): [] for c in conds}); gl = []
        for s_ in CFG["seeds"]:
            net = train_one(mode, use, D, s_, md)
            acc[("real", "val")].append(predict(net, D, "val", "real"))
            for c in conds: acc[(c, "test")].append(predict(net, D, "test", c))
            if mode == "gate": gl.append(np.stack(gate_values(net, D)))
        for k, v in acc.items(): P[(label, *k)] = np.mean(v, 0)
        if gl: gv[label] = np.mean(gl, 0)                                              # (2, n_test): audio gate, text gate
        print(f"  {label:28s} done ({time.time() - t0:.0f}s)")
    # late fusion: weights for the audio and text probabilities tuned on val (coarse grid)
    yv = y[D["val_np"]]; lv = lambda k: np.log(P[(k, "real", "val")] + 1e-8); best = {}
    for wa in (0, .25, .5, .75, 1):
        for wt in (0, .25, .5, .75, 1):
            f = f1_score(yv, (lv("video only") + wa * lv("audio only") + wt * lv("text only")).argmax(1), average="macro")
            if f > best.get("f", -1) + 1e-9: best = dict(f=f, wa=wa, wt=wt)
    wa, wt = best["wa"], best["wt"]; lab = f"late V+A+T (wa={wa}, wt={wt})"
    lt = lambda k, c="real": np.log(P[(k, c, "test")] + 1e-8)
    P[(lab, "real", "test")] = softmax_np(lt("video only") + wa * lt("audio only") + wt * lt("text only"))
    P[(lab, "noA", "test")] = softmax_np(lt("video only") + wt * lt("text only"))
    P[(lab, "noT", "test")] = softmax_np(lt("video only") + wa * lt("audio only"))
    P[(lab, "noAT", "test")] = softmax_np(lt("video only"))
    P[(lab, "shufA", "test")] = softmax_np(lt("video only") + wa * lt("audio only", "shufA") + wt * lt("text only"))
    P[(lab, "shufT", "test")] = softmax_np(lt("video only") + wa * lt("audio only") + wt * lt("text only", "shufT"))
    return D, P, gv, yt, [s[0] for s in SPECS] + [lab]

In [8]:
# Cell 6: results tables
def report(name, D, P, gv, yt, methods):
    te = D["test_np"]; base = P[("video only", "real", "test")]; rows = []
    for m in methods:
        pt, ci = with_ci(yt, P[(m, "real", "test")])
        dm = "" if m == "video only" else "{:+.3f} [{:+.3f}, {:+.3f}]".format(*paired(yt, base, P[(m, "real", "test")]))
        rows.append(dict(method=m, macro_F1=fmt(pt, ci, "macro_F1"), macro_AUC=fmt(pt, ci, "macro_AUC"), top2=f"{pt['top2_acc']:.3f}",
                         diff_vs_video=dm, **{c: round(pt[f"rec_{c}"], 2) for c in CLASS_NAMES}))
    main = pd.DataFrame(rows); pd.set_option("display.width", 260, "display.max_colwidth", 50)
    print(f"\n===== {name}: main results (mean of {len(CFG['seeds'])} seeds; diff = macro-F1 minus video only) =====")
    print(main.to_string(index=False)); main.to_csv(f"{RES}/fusion3_main_{name}.csv", index=False)

    rows = []
    for m in methods:
        if m in ("video only", "audio only", "text only"): continue
        r = dict(method=m)
        for c in ("real", "noA", "noT", "noAT", "shufA", "shufT"):
            if (m, c, "test") in P: r[c] = round(f1_score(yt, P[(m, c, "test")].argmax(1), average="macro"), 3)
        rows.append(r)
    ctl = pd.DataFrame(rows)
    print(f"\n{name}: controls (macro-F1). noA/noT/noAT = that modality removed, shufA/shufT = taken from a different-class clip")
    print(ctl.to_string(index=False)); ctl.to_csv(f"{RES}/fusion3_controls_{name}.csv", index=False)

    show = ["video only", "text only", "early V+T", "early V+A+T", "gated V+A+T (router)", "cross-attention V+A+T", methods[-1]]
    for title, garr, gnames in (("audio type", grp[te], ("music-dominant", "speech present", "other audio")),
                                ("text availability", tgrp[te], ("speech + on-screen text", "one text source", "no text"))):
        rows = []
        for gname in gnames:
            sel = garr == gname
            if sel.sum() < 20: continue
            rows.append(dict(group=gname, n=int(sel.sum()), **{m: round(f1_score(yt[sel], P[(m, "real", "test")][sel].argmax(1), average="macro"), 3) for m in show}))
        byg = pd.DataFrame(rows); print(f"\n{name}: macro-F1 by {title} (small groups: treat differences of a few points as noise)")
        print(byg.to_string(index=False)); byg.to_csv(f"{RES}/fusion3_by_{title.replace(' ', '_')}_{name}.csv", index=False)

    if "gated V+A+T (router)" in gv:
        ga_, gt_ = gv["gated V+A+T (router)"]
        print(f"\n{name}: mean gate values on test clips (audio gate by audio type, text gate by text availability)")
        print({k: round(float(ga_[grp[te] == k].mean()), 3) for k in ("music-dominant", "speech present", "other audio") if (grp[te] == k).any()})
        print({k: round(float(gt_[tgrp[te] == k].mean()), 3) for k in ("speech + on-screen text", "one text source", "no text") if (tgrp[te] == k).any()})

D_m, P_m, gv_m, yt_m, meth_m = run_set("matched", in_matched)
report("matched", D_m, P_m, gv_m, yt_m, meth_m)


##### matched: train=1004 val=104 test=395 #####
  video only                   done (10s)
  audio only                   done (18s)
  text only                    done (25s)
  early V+A                    done (35s)
  early V+T                    done (45s)
  early V+A+T                  done (57s)
  gated V+A+T (router)         done (72s)
  cross-attention V+A+T        done (90s)
  gated V+A+T, no mod-drop     done (105s)

===== matched: main results (mean of 5 seeds; diff = macro-F1 minus video only) =====
                   method             macro_F1            macro_AUC  top2           diff_vs_video  Safe  Harmful  Adult  Suicide
               video only 0.833 [0.796, 0.868] 0.947 [0.929, 0.965] 0.919                          0.93     0.71   0.81     0.90
               audio only 0.569 [0.519, 0.616] 0.793 [0.761, 0.824] 0.767 -0.265 [-0.320, -0.207]  0.52     0.49   0.55     0.73
                text only 0.538 [0.486, 0.584] 0.774 [0.743, 0.802] 0.790 -0.297 [-0.353, -0.242]

In [9]:
# Cell 7: secondary set (full clean set)
D_f, P_f, gv_f, yt_f, meth_f = run_set("full", np.ones(N, bool))
report("full", D_f, P_f, gv_f, yt_f, meth_f)
print(f"\nSaved tables to {RES}")


##### full: train=2684 val=336 test=870 #####
  video only                   done (18s)
  audio only                   done (37s)
  text only                    done (56s)
  early V+A                    done (82s)
  early V+T                    done (107s)
  early V+A+T                  done (139s)
  gated V+A+T (router)         done (179s)
  cross-attention V+A+T        done (226s)
  gated V+A+T, no mod-drop     done (265s)

===== full: main results (mean of 5 seeds; diff = macro-F1 minus video only) =====
                    method             macro_F1            macro_AUC  top2           diff_vs_video  Safe  Harmful  Adult  Suicide
                video only 0.863 [0.839, 0.885] 0.964 [0.954, 0.973] 0.941                          0.93     0.75   0.82     0.94
                audio only 0.673 [0.642, 0.706] 0.869 [0.851, 0.889] 0.843 -0.190 [-0.223, -0.158]  0.72     0.57   0.60     0.80
                 text only 0.570 [0.539, 0.603] 0.814 [0.794, 0.834] 0.803 -0.293 [-0.330, -0.25

In [10]:
# Cell 8: export predictions for the policy-layer notebook
def export_predictions(D, P, name, fused="cross-attention V+A+T"):
    te = D["test_np"]; out = pd.DataFrame({"uid": order.uid.values[te], "split": "test", "label": y[te]})
    for k, c in enumerate(["safe", "harmful", "adult", "suicide"]):
        out[f"p_video_{c}"] = P[("video only", "real", "test")][:, k]; out[f"p_fused_{c}"] = P[(fused, "real", "test")][:, k]
    out["aud_ok"] = aud_ok[te]; out["has_asr"] = Mt[te, 0]; out["has_ocr"] = Mt[te, 1]
    out.to_csv(f"{RES}/predictions_{name}.csv", index=False); print(f"saved {RES}/predictions_{name}.csv ({len(out)} test clips)")
export_predictions(D_f, P_f, "full"); export_predictions(D_m, P_m, "matched")

saved /kaggle/working/results/predictions_full.csv (870 test clips)
saved /kaggle/working/results/predictions_matched.csv (395 test clips)


In [11]:
!zip -r results_final3_final.zip /kaggle/working/results

  adding: kaggle/working/results/ (stored 0%)
  adding: kaggle/working/results/fusion3_main_matched.csv (deflated 57%)
  adding: kaggle/working/results/predictions_full.csv (deflated 59%)
  adding: kaggle/working/results/fusion3_main_full.csv (deflated 57%)
  adding: kaggle/working/results/predictions_matched.csv (deflated 58%)
  adding: kaggle/working/results/fusion3_by_text_availability_full.csv (deflated 39%)
  adding: kaggle/working/results/fusion3_controls_matched.csv (deflated 50%)
  adding: kaggle/working/results/fusion3_by_text_availability_matched.csv (deflated 39%)
  adding: kaggle/working/results/fusion3_by_audio_type_matched.csv (deflated 36%)
  adding: kaggle/working/results/fusion3_controls_full.csv (deflated 48%)
  adding: kaggle/working/results/fusion3_by_audio_type_full.csv (deflated 35%)


In [12]:
import os
import pandas as pd

RES = "/kaggle/working/results"

for filename in [
    "predictions_full.csv",
    "predictions_matched.csv"
]:
    path = os.path.join(RES, filename)

    print(f"\n{filename}")
    print("Exists:", os.path.isfile(path))

    if os.path.isfile(path):
        df = pd.read_csv(path)
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        display(df.head(3))


predictions_full.csv
Exists: True
Shape: (870, 14)
Columns: ['uid', 'split', 'label', 'p_video_safe', 'p_fused_safe', 'p_video_harmful', 'p_fused_harmful', 'p_video_adult', 'p_fused_adult', 'p_video_suicide', 'p_fused_suicide', 'aud_ok', 'has_asr', 'has_ocr']


,uid,split,label,p_video_safe,p_fused_safe,p_video_harmful,p_fused_harmful,p_video_adult,p_fused_adult,p_video_suicide,p_fused_suicide,aud_ok,has_asr,has_ocr
0,4eb5410b5075,test,2,0.014358,0.005487,0.037597,0.011246,0.928437,0.976231,0.019608,0.007035,1.0,0.0,0.0
1,08e7e4c6f151,test,2,0.032474,0.006113,0.032555,0.004865,0.874163,0.979363,0.060808,0.009659,1.0,0.0,0.0
2,69087619caeb,test,2,0.143523,0.065849,0.051065,0.020043,0.644517,0.902250,0.160896,0.011858,1.0,1.0,1.0



predictions_matched.csv
Exists: True
Shape: (395, 14)
Columns: ['uid', 'split', 'label', 'p_video_safe', 'p_fused_safe', 'p_video_harmful', 'p_fused_harmful', 'p_video_adult', 'p_fused_adult', 'p_video_suicide', 'p_fused_suicide', 'aud_ok', 'has_asr', 'has_ocr']


,uid,split,label,p_video_safe,p_fused_safe,p_video_harmful,p_fused_harmful,p_video_adult,p_fused_adult,p_video_suicide,p_fused_suicide,aud_ok,has_asr,has_ocr
0,69087619caeb,test,2,0.243653,0.157095,0.083418,0.016708,0.434550,0.654525,0.238379,0.171672,1.0,1.0,1.0
1,3cc20cc86f6d,test,2,0.021173,0.010206,0.022653,0.006901,0.941995,0.976790,0.014179,0.006103,1.0,0.0,0.0
2,f800c2aa5c39,test,2,0.022225,0.004235,0.127502,0.010559,0.738353,0.974554,0.111920,0.010652,1.0,1.0,1.0
